# Ingestão Bronze: ADLS Gen2 (raw) ➔ Delta Lake (squad1/bronze)
Este notebook realiza:
- Autenticação Spark no ADLS via opções da leitura/escrita (credenciais somente via `.env`).
- Polling de novos arquivos `ecommerce_enderecos.parquet` em `raw/real-time-data` (micro-lote = 1 arquivo).
- Leitura do `.parquet` direto com Spark, **sem nenhuma transformação** nos dados (apenas padronização de tipo em colunas 100% nulas, ver célula 3).
- Colunas de auditoria: `bronze_ingested_at` (`current_timestamp()`) e `bronze_source_file` (arquivo de origem).
- Escrita Delta (`append`) em `squad1/bronze/ecommerce_enderecos`, particionada por data de ingestão (`ano`, `mes`, `dia`, `hora`).

In [ ]:
import os
import time
from dotenv import load_dotenv
from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F, types as T

# ==============================================================================
# 1. Carregamento Seguro das Credenciais e Parâmetros
# ==============================================================================
load_dotenv(".env")

client_id            = os.getenv("ADLS_CLIENT_ID")
tenant_id            = os.getenv("ADLS_TENANT_ID")
client_secret        = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
source_container     = os.getenv("ADLS_CONTAINER_NAME", "raw")

assert all([client_id, tenant_id, client_secret, storage_account_name]), "Faltou variável do ADLS no arquivo .env."

# ==============================================================================
# 2. Autenticação Spark no ADLS Gen2
# ==============================================================================
# O compute serverless bloqueia spark.conf.set("fs.azure..."), então as credenciais
# são passadas como opções de cada leitura/escrita.
adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

source_glob = f"abfss://{source_container}@{storage_account_name}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_enderecos.parquet"
squad_base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
bronze_path = f"{squad_base}/bronze/ecommerce_enderecos"
poll_interval_seconds = 15


def ler_delta(path):
    """Lê uma tabela Delta do ADLS. Retorna None apenas se ela ainda não foi criada."""
    try:
        df = spark.read.format("delta").options(**adls_opts).load(path)
        df.schema  # força a resolução do caminho (a leitura é lazy)
        return df
    except AnalysisException as e:
        if any(m in str(e) for m in ("PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST", "DELTA_MISSING_DELTA_TABLE", "is not a Delta table")):
            return None
        raise


# Schema esperado da origem. Alguns lotes gravam colunas 100% nulas como int
# (complemento, bairro, cep, cidade), o que quebraria o append no Delta.
target_schema = T.StructType.fromDDL(
    "id_endereco bigint, id_cliente bigint, logradouro string, numero bigint, "
    "complemento string, bairro string, cep string, cidade string, estado string, "
    "latitude double, longitude double, apelido string, is_principal boolean"
)

print("✔ Parâmetros carregados.")

In [ ]:
# ==============================================================================
# 3. Checkpoint: arquivos já presentes na Bronze
# ==============================================================================
# Sem try/except de propósito: se a Bronze existir mas não puder ser lida, o notebook
# deve parar, e não seguir com o checkpoint vazio (o que reingeriria tudo em duplicidade).
df_bronze_atual = ler_delta(bronze_path)

if df_bronze_atual is None:
    control_checkpoint = set()
    print("ℹ Bronze ainda não existe em squad1/bronze/ecommerce_enderecos. Será criada na primeira ingestão.")
else:
    control_checkpoint = {
        r["bronze_source_file"]
        for r in df_bronze_atual.select("bronze_source_file").distinct().collect()
    }
    print(f"✔ Histórico carregado: {len(control_checkpoint)} arquivo(s) previamente ingerido(s).")

In [ ]:
# ==============================================================================
# 4. Loop de Monitoramento: micro-lote Bronze (sem transformação)
# ==============================================================================
def listar_arquivos_origem():
    """Caminhos de todos os ecommerce_enderecos.parquet em raw/real-time-data, em ordem cronológica."""
    try:
        arquivos = spark.read.format("binaryFile").options(**adls_opts).load(source_glob).select("path").collect()
    except AnalysisException as e:
        if "PATH_NOT_FOUND" in str(e):
            return []
        raise
    # As pastas seguem AAAA/MM/DD/HHMMSS, então a ordem alfabética é a cronológica
    return sorted(r.path for r in arquivos)


def ler_micro_lote(file_path):
    """Lê o .parquet com Spark mantendo os dados originais.

    O único ajuste é o cast para o schema esperado, e ele só é permitido quando a
    coluna divergente está 100% nula. Qualquer outra diferença de schema interrompe
    o micro-lote para decisão manual, em vez de alterar ou descartar dado real.
    """
    df = spark.read.options(**adls_opts).parquet(file_path)

    esperadas = [c.name for c in target_schema.fields]
    faltando = set(esperadas) - set(df.columns)
    extras = set(df.columns) - set(esperadas)
    if faltando or extras:
        raise ValueError(f"Schema diferente do esperado. Faltando: {sorted(faltando)} | Extras: {sorted(extras)}")

    for campo in target_schema.fields:
        tipo_origem = df.schema[campo.name].dataType
        if tipo_origem != campo.dataType and df.filter(F.col(campo.name).isNotNull()).limit(1).count() > 0:
            raise ValueError(f"Coluna '{campo.name}' veio como {tipo_origem} com valores preenchidos. Revisar antes de ingerir.")

    return df.select([F.col(c.name).cast(c.dataType) for c in target_schema.fields])


print(f"Iniciando monitoramento de 'ecommerce_enderecos.parquet' em '{source_container}/real-time-data'...\n")

try:
    while True:
        try:
            new_files = [p for p in listar_arquivos_origem() if p not in control_checkpoint]
        except Exception as e:
            print(f"⚠ Falha ao listar a origem, nova tentativa em {poll_interval_seconds}s: {e}")
            time.sleep(poll_interval_seconds)
            continue

        # try/except por arquivo: um arquivo com problema não trava os seguintes
        for file_path in new_files:
            try:
                print(f"Processando: {file_path}")

                df_bronze = (
                    ler_micro_lote(file_path)
                    # Apenas metadados de auditoria; os dados originais permanecem intactos
                    .withColumn("bronze_ingested_at", F.current_timestamp())
                    .withColumn("bronze_source_file", F.lit(file_path))
                    # Partições derivadas da data de ingestão
                    .withColumn("ano",  F.year("bronze_ingested_at"))
                    .withColumn("mes",  F.month("bronze_ingested_at"))
                    .withColumn("dia",  F.dayofmonth("bronze_ingested_at"))
                    .withColumn("hora", F.hour("bronze_ingested_at"))
                )

                (
                    df_bronze.write
                    .format("delta")
                    .mode("append")
                    .options(**adls_opts)
                    .partitionBy("ano", "mes", "dia", "hora")
                    .save(bronze_path)
                )

                control_checkpoint.add(file_path)
                print(f"✔ Sucesso: ingerido na Bronze ({df_bronze.count()} registros).")

            except Exception as e:
                # Não entra no checkpoint: será tentado de novo na próxima varredura
                print(f"✖ Falha em '{file_path}': {e}")

        time.sleep(poll_interval_seconds)

except KeyboardInterrupt:
    print("\nProcesso interrompido manualmente pelo usuário.")